# Homework 7

In Homework 5, you solved Problem 5.1 from Loucks and van Beek (2017) using Deterministic Dynamic Programming. Here you'll repeat the same assignment with two versions of Stochastic Dynamic Programming: one with risk neutral constraints and one with risk averse constraints. To do this, we'll assume log-space seasonal inflows follow a bivariate normal distribution, different from Homework 5. The modified problem is described below, with differences from Homework 5 indicated in bold.

A reservoir serves multiple purposes. It is a recreation site for swimmers, wind surfers and boaters; a flood storage reservoir; and an irrigation supply structure. Each season, there is a required minimum release for irrigation to meet water rights downstream. Reservoir releases in excess of the irrigation requirement flow to a wetland downstream. The wetland also receives return flows from irrigation, equal to 30\% of the water diverted for irrigation. These return flows are highly saline, which can damage the ecosystem. Different target water allocations are favorable for each of these objectives. Based on mean inflows across 4 seasons, your task is to determine how much water should be released from the reservoir in each season to minimize the sum of percent deviation across all targets given the following parameters and targets:

* Reservoir storage capacity, $K$: 30 million m$^3$ (mcm)
* Maximum reservoir release, $Rmax$: 50 mcm
* **Reservoir seasonal inflows distribution, $Y=\ln(Qin)$: $N(1.8, 0.1)$, $N(3.6, 0.3)$, $N(2.9, 0.3)$, $N(2.3, 0.2)$ ($Qin$ in mcm)**
* **Correlation between consecutive season's log-space flows: $\rho=0.7$**
* Minimum releases for irrigation each season, $Qirr$: 5, 20, 10, 5 mcm
* Salinity concentration of reservoir water, $Cres$: 1 part per trillion (ppt)
* Salinity concentration of irrigation return flow water, $Cirr$: 20 ppt
* Target salinity concentration in the wetland, $Cwet\_maxTarget$: 3 ppt
* Target minimum flow in the wetland each season, $Qwet\_minTarget$: 10, 20, 15, 15 mcm
* Target reservoir storages each period, $S$: 20, 5, 20, 20 mcm

## Part a (10 pts)

Re-run your dynamic program from HW 5 to find the optimal releases from the reservoir each season for discrete storage states between 0 and 30 in increments of 5, assuming the mean inflow each season. Use the same cost function as HW5: the sum of percent deviations from all targets, where storages above and below the target are penalized but only wetland salinity concentrations below the minimum target are penalized, and only wetland flows above the maximum target.

$\color{red}{\text{Import libraries and define parameters. Code below is complete.}}$

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.optimize import minimize

# inflow parameters
muY = np.array([1.8, 3.6, 2.9, 2.3])
sigmaY = np.array([0.1, 0.3, 0.3, 0.2])
nSeasons = len(muY)
meanQ = np.exp(muY + 0.5*sigmaY**2) # do not repeat first season as last
rho = 0.7 # correlation between consecutive log-space flows

# define model parameters
# reservoir parameters
K = 30
maxR = 50
S_target = np.array([20, 5, 20, 20])

# irrigation parameters
Qirr = np.array([5,20,10,5])
Qirr_RF =  0.3

# salinity parameters
Cres = 1
Cirr = 20
Cwet_maxTarget = 5

# wetland parameters
Qwet_minTarget = np.array([10,20,15,15])

$\color{red}{\text{Complete these functions in the code chunk below by copying (and editing if necessary) your code from Homework 5.}}$

In [ ]:
######################## DP Optimization ########################

def findBestR_DDP(R, sCurrent, States, S_target, Cwet_maxTarget, Qwet_minTarget, Q, Qirr, FutureCost):

  return TotalCost

def calcCostDDP(S, Q, Qirr, S_target, Cwet_maxTarget, Qwet_minTarget, bounds, FutureCost):

  return Rbest, Cbest

Now use the following code chunk to perform discrete DDP with the above functions, solving for the optimal release each season from storage levels between 0 and 30 in increments of 5 assuming the mean inflow each season. Exit the loop if an entire cycle has an average percent difference below 0.1% each season. Report a table with the prescribed release each season from each storage state. $\color{red}{\text{The code below is complete.}}$

In [ ]:
# get indices of stages
nStages = len(meanQ)
forward_indices = np.arange(nStages)
backward_indices = forward_indices[::-1]
backward_indices = np.insert(backward_indices,0,0)

# discretize states
states = np.arange(0,31,5)
nStates = len(states)

# bounds on release decisions each season
bounds = np.empty([nSeasons,2])
for i in range(nSeasons):
  bounds[i,:] = [Qirr[i],maxR]

# initialize matrices with costs of each state at each stage
# and optimal releases to make from each state at each stage
DDP_costs = np.empty([nStates,nStages])
DDP_release_policy = np.empty([nStates,nStages])

# initialize FutureCost at 0 for all states; will update as we move backwards
FutureCost = np.zeros([nStates])

# begin backward-moving DP
loop = True
tolerance = 0.1 # average % difference in optimal releases from one cycle to the next at which to stop looping
avgPctDiff = np.inf
cycle = 0 # number of times cycling through all seasons
while loop:
  count = 0
  cycle += 1
  print("Cycle %d" % cycle)
  for index in backward_indices[0:-1]:
    # find optimal release and value of each state in this stage
    R, FutureCost = calcCostDDP(states, meanQ[index-1], Qirr[index-1],
                               S_target[index], Cwet_maxTarget, Qwet_minTarget[index-1],
                               bounds[index-1], FutureCost)

    # find average % difference in optimal releases at this stage compared to the last loop
    if cycle > 1:
      avgPctDiff = np.mean(np.abs(R - DDP_release_policy[:,index-1])*100 / DDP_release_policy[:,index-1])
      print(avgPctDiff)
      if avgPctDiff < tolerance:
        count += 1

    # update best releases and value of each state
    DDP_costs[:,index] = FutureCost
    DDP_release_policy[:,index-1] = R

  # stop loop if no change in optimal decisions across all iterations
  if count == len(backward_indices[0:-1]):
      break

release_policy_df = pd.DataFrame(DDP_release_policy,
                                 columns=["Season 1","Season 2","Season 3","Season 4"],
                                 index=states)
release_policy_df.index.rename("Storage",inplace=True)
release_policy_df

## Part b (10 pts)

Now explicitly consider uncertainty in the optimization using stochastic dynamic programming. First, compute the transition probabilities from 15 discrete log-space flow levels $Y_{t-1}$ in season $t-1$ to 15 discrete log-space flow levels $Y_t$ in season $t$. Assume consecutive log-space flows are bivariate normal. Print the transition probabilities each season. $\color{red}{\text{The code below is complete; it's the same as the code from class.}}$

In [ ]:
forward_indices = np.append(forward_indices,0)

# find transition probabilities from discrete flow values in one season to the next
def calcTransProb(mu, sigma, rho, nLevels):
  transprob = np.empty([nLevels,nLevels])
  # discrete levels of log-space flows
  Ylevels1 = np.linspace(ss.norm.ppf(0.01,mu[0],sigma[0]),ss.norm.ppf(0.99,mu[0],sigma[0]),nLevels)
  Ylevels2 = np.linspace(ss.norm.ppf(0.01,mu[1],sigma[1]),ss.norm.ppf(0.99,mu[1],sigma[1]),nLevels)
  print("Log-space flow levels, t-1: ", Ylevels1)
  print("Log-space flow levels, t: ", Ylevels2)
  for i in range(nLevels):
    # find conditional distribution of Qlevels2[j] given flow is Qlevels1[i] in previous season
    mu_cond = mu[1] + rho*(sigma[1]/sigma[0])*(Ylevels1[i] - mu[0])
    sigma_cond = sigma[1] * np.sqrt((1-rho**2))
    for j in range(nLevels):
      transprob[i,j] = ss.norm.pdf(Ylevels2[j], mu_cond, sigma_cond)

    #normalize probabilities to sum to 1
    transprob[i,:] = transprob[i,:] / np.sum(transprob[i,:])

  return transprob

# discretize inflows into nLevels and find transition probabilities between them
nLevels=15
transprob = []
for i, index in enumerate(forward_indices[0:-1]):
  print("Season " + str(index+1) + " to Season " + str(forward_indices[i+1]+1))
  mu = np.array([muY[index],muY[forward_indices[i+1]]]) # log-space mean
  sigma = np.array([sigmaY[index],sigmaY[forward_indices[i+1]]]) # log-space standard deviation
  transprob.append(calcTransProb(mu, sigma, rho, nLevels))
  print("Transition probabilities: ", transprob[i])
  print("\n")

$\color{red}{\text{Make a heatmap of the transition probabilities across the 4 seasons in a 2x2 panel figure in the code chunk below.}}$  
Hint: add `ax=ax[:,-1]` to `fig.colorbar()` so the colorbar spans both rows of figures.

## Part c (20 pts - 10 pts per function)

Next, modify the functions $\texttt{calcCostSDP}$ and $\texttt{findBestR}$ from the [SDPexample.ipynb](https://colab.research.google.com/github/EnvSystemsUVA/CodingExamples/blob/main/10_SDPexample.ipynb) notebook shown in class to calculate the new cost function, the same as in part (a). Note $\texttt{calcCostSDP}$ includes an argument for whether the minimum irrigation release and maximum reservoir releases are chance constraints or not. If $\texttt{chance_constraint}=True$, ensure the maximum reservoir release is not exceeded even under the 99th percentile inflow and the release is not below the minimum irrigation flow even under the 1st percentile inflow. If $\texttt{chance_constraint}=False$, have the constraints only apply to the mean inflow.

In [ ]:
######################## SDP Optimization ########################
from scipy.optimize import minimize

def findBestR_SDP(R, s, j, S, Ylevels2, transprob, Qirr,
                  S_target, Cwet_maxTarget, Qwet_minTarget, FutureExpCost):


  return C

def calcCostSDP(S, Ylevels1, Ylevels2, Rguess, transprob, Qirr,
                S_target, Cwet_maxTarget, Qwet_minTarget,
                FutureExpCost, chance_constraint=False):


  return Rbest, Cbest

## Part d (10 pts)

Now run backward-moving SDP with the code chunk below, which uses the functions you wrote above to solve for the optimal release from each combination of 7 storage levels and 15 previous inflow levels WITHOUT a chance constraint. This code stops when the average difference in the release decisions at each state change by less than $0.5\%$ in a given stage from one cycle through all seasons to the next. $\color{red}{\text{The code below is complete.}}$

In [ ]:
# use same stages, states, indices and bounds as for DDP
# initialize matrices with costs of each state at each stage
# and optimal releases to make from each state at each stage
SDP1_costs = np.empty([nStates, nLevels, nStages])
SDP1_release_policy = np.empty([nStates, nLevels, nStages])

# initialize FutureCost at 0 for all states; will update as we move backwards
FutureExpCost1 = np.zeros([nStates, nLevels])

# begin backward-moving SDP
tolerance = 0.5# average % difference in optimal releases from one cycle to the next at which to stop looping
avgPctDiff = np.inf
cycle = 0 # number of times cycling through all seasons
while loop:
  count = 0
  cycle +=1
  print("Cycle %d" % cycle)
  for index in backward_indices[0:-1]:
    mu = np.array([muY[index-2],muY[index-1]])
    sigma = np.array([sigmaY[index-2],sigmaY[index-1]])
    Ylevels1 = np.linspace(ss.norm.ppf(0.01,mu[0],sigma[0]),ss.norm.ppf(0.99,mu[0],sigma[0]),nLevels)
    Ylevels2 = np.linspace(ss.norm.ppf(0.01,mu[1],sigma[1]),ss.norm.ppf(0.99,mu[1],sigma[1]),nLevels)

    # find optimal release and value of each state in this stage
    # states (storage targets) are for next period (index) while inflows and releases are for this period (index-1)
    R1, FutureExpCost1 = calcCostSDP(states, Ylevels1, Ylevels2, meanQ[index-1], transprob[index-1], Qirr[index-1],
                                   S_target[index], Cwet_maxTarget, Qwet_minTarget[index-1], FutureExpCost1)

    # find average % difference in optimal releases at this stage compared to the last loop
    if cycle > 1:
      avgPctDiff = np.nanmean(np.abs(R1 - SDP1_release_policy[:,:,index-1])*100/SDP1_release_policy[:,:,index-1])
      print(avgPctDiff)
      if avgPctDiff < tolerance:
        count += 1

    # update best releases and value of each state
    SDP1_costs[:,:,index] = FutureExpCost1
    SDP1_release_policy[:,:,index-1] = R1
    #print(SDP_release_policy[:,:,index-1])

  # stop loop if average % change in optimal decisions across all iterations < tolerance
  if count == len(backward_indices[0:-1]):
    break

# print the release policy of each stage
for i in range(nSeasons):
  cols = []
  Ylevels = np.linspace(ss.norm.ppf(0.01,muY[i-1],sigmaY[i-1]),ss.norm.ppf(0.99,muY[i-1],sigmaY[i-1]),nLevels)
  for j in range(nLevels):
    cols.append("Q = %0.2f" % np.exp(Ylevels[j]))
  SDP1_release_policy_df = pd.DataFrame(SDP1_release_policy[:,:,i], columns=cols,index=states)
  SDP1_release_policy_df.index.rename("Storage",inplace=True)
  print("Season " + str(i+1) + " release policy", SDP1_release_policy_df)

$\color{red}{\text{Make a heatmap of the releases as a function of the current storage and previous inflow across the 4 seasons in a 2x2 panel figure.}}$  
Again, add `ax=ax[:,-1]` to `fig.colorbar()` to make the colorbar span both rows.

$\color{red}{\text{What do you observe and why?}}$

## Part e (10 pts)

Repeat part d WITH a chance constraint. $\color{red}{\text{The code below is complete.}}$

In [ ]:
# use same stages, states, indices and bounds as for DDP
# initialize matrices with costs of each state at each stage
# and optimal releases to make from each state at each stage
SDP2_costs = np.empty([nStates, nLevels, nStages])
SDP2_release_policy = np.empty([nStates, nLevels, nStages])

# initialize FutureCost at 0 for all states; will update as we move backwards
FutureExpCost2 = np.zeros([nStates, nLevels])

# begin backward-moving SDP
tolerance = 0.5# average % difference in optimal releases from one cycle to the next at which to stop looping
avgPctDiff = np.inf
cycle = 0 # number of times cycling through all seasons
while loop:
  count = 0
  cycle +=1
  print("Cycle %d" % cycle)
  for index in backward_indices[0:-1]:
    mu = np.array([muY[index-2],muY[index-1]])
    sigma = np.array([sigmaY[index-2],sigmaY[index-1]])
    Ylevels1 = np.linspace(ss.norm.ppf(0.01,mu[0],sigma[0]),ss.norm.ppf(0.99,mu[0],sigma[0]),nLevels)
    Ylevels2 = np.linspace(ss.norm.ppf(0.01,mu[1],sigma[1]),ss.norm.ppf(0.99,mu[1],sigma[1]),nLevels)

    # find optimal release and value of each state in this stage
    # states (storage targets) are for next period (index) while inflows and releases are for this period (index-1)
    R2, FutureExpCost2 = calcCostSDP(states, Ylevels1, Ylevels2, meanQ[index-1], transprob[index-1], Qirr[index-1],
                                    S_target[index], Cwet_maxTarget, Qwet_minTarget[index-1],
                                    FutureExpCost2, chance_constraint=True)

    # find average % difference in optimal releases at this stage compared to the last loop
    if cycle > 1:
      avgPctDiff = np.nanmean(np.abs(R2 - SDP2_release_policy[:,:,index-1])*100/SDP2_release_policy[:,:,index-1])
      print(avgPctDiff)
      if avgPctDiff < tolerance:
        count += 1

    # update best releases and value of each state
    SDP2_costs[:,:,index] = FutureExpCost2
    SDP2_release_policy[:,:,index-1] = R2
    #print(SDP_release_policy[:,:,index-1])

  # stop loop if average % change in optimal decisions across all iterations < tolerance
  if count == len(backward_indices[0:-1]):
    break

# print the release policy of each stage
for i in range(nSeasons):
  cols = []
  Ylevels = np.linspace(ss.norm.ppf(0.01,muY[i-1],sigmaY[i-1]),ss.norm.ppf(0.99,muY[i-1],sigmaY[i-1]),nLevels)
  for j in range(nLevels):
    cols.append("Q = %0.2f" % np.exp(Ylevels[j]))
  SDP2_release_policy_df = pd.DataFrame(SDP2_release_policy[:,:,i], columns=cols,index=states)
  SDP2_release_policy_df.index.rename("Storage",inplace=True)
  print("Season " + str(i+1) + " release policy", SDP2_release_policy_df)

$\color{red}{\text{Make a heatmap of the releases as a function of the current storage and previous inflow across the 4 seasons in a 2x2 panel figure.}}$  
Again, add `ax=ax[:,-1]` to `fig.colorbar()` to make the colorbar span both rows.

$\color{red}{\text{How does this chance-constrained release policy differ from the version with a mean constraint and why?}}$

## Part f (10 pts)

Simulate 50 years of operations in which the release each season is determined by the DDP policy found in part (a). Repeat this using the SDP operating policies found to be optimal without chance constraints in part (d) and with chance constraints in part (e). In each case, if there is insufficient water to meet the release prescribed by the policy, only release as much water as is available. Likewise, if the prescribed release would result in exceeding the reservoir capacity, release as much as needed to prevent that (this may not meet the irrigation requirement, or may exceed maxR, but that's okay for the purpose of this simulation).

$\color{red}{\text{Complete the code below using your functions from Homework 5.}}$
$\color{red}{\text{Modify $\texttt{getSimStates}$ to count the number of violations of the minimum release Qirr and maximum release maxR.}}$
$\color{red}{\text{$\texttt{getSimCosts}$ will be unchanged}}$.

In [ ]:
######################## Simulation ########################

# initialize storages and releases for simulation of 50 years of 4 seasons with DDP and SDP policies
nYears = 50
nSeasons = 4

class Solution():
  # initialize Solution class with certain attributes for DDP vs. SDP solutions
  def __init__(self):
    self.simS = np.zeros([nYears,nSeasons])
    self.simR = np.zeros([nYears,nSeasons])
    self.simQwet = np.zeros([nYears,nSeasons])
    self.simCwet = np.zeros([nYears,nSeasons])
    self.S_costs = np.zeros([nYears])
    self.Qwet_costs = np.zeros([nYears])
    self.Cwet_costs = np.zeros([nYears])
    self.Total_costs = np.zeros([nYears])
    self.prescribedR = None
    self.Rmin_violations = 0
    self.Rmax_violations = 0

  # method of Solution class to calculate simulated R, S and min/max release violations
  def getSimStates(self, Q, year, season):


  # method of Solution class to calculate cost (total deviation from targets) over simulation
  def getSimCosts(self, year):


Now run the code below to simulate the three policies. $\color{red}{\text{The code below is complete.}}$

In [ ]:
from scipy.interpolate import RegularGridInterpolator as interp2d

# create objects of Solution class for DDP and SDP solutions
DDP = Solution()
SDP1 = Solution()
SDP2 = Solution()

# start at target storage
DDP.simS[0,0] = S_target[0]
SDP1.simS[0,0] = S_target[0]
SDP2.simS[0,0] = S_target[0]

# vector of standard normal random variables for flow simulation
Z = np.zeros([nYears*nSeasons+1])

# generate prior season's random normal inflow
seed = 0
Z[seed] = ss.norm.rvs(0,1,1)[0]
Qpast = np.exp(Z[seed-1]*sigmaY[-1] + muY[-1])

# simulate operations over 50 years of 4 seasons
for year in range(nYears):
  for season in range(nSeasons):
    # generate inflow (set a seed to make it reproducible)
    seed += 1
    np.random.seed(seed)
    # generate flow conditional on previous season's flow and transform to real-space
    Z[seed] = rho*(Z[seed-1]) + ss.norm.rvs(0,1,1)[0]*np.sqrt(1-rho**2)
    Qnow = np.exp(Z[seed]*sigmaY[season] + muY[season])

    # find DDP releases from its policy: interpolate release between nearest storages
    DDP.prescribedR = np.interp(DDP.simS[year,season],states,DDP_release_policy[:,season])

    # find SDP releases from its policies: interpolate release between nearest storages and flows
    Ylevels1 = np.linspace(ss.norm.ppf(0.01,muY[season-1],sigmaY[season-1]),ss.norm.ppf(0.99,muY[season-1],sigmaY[season-1]),nLevels)
    f1 = interp2d((states, Ylevels1), SDP1_release_policy[:,:,season])
    f2 = interp2d((states, Ylevels1), SDP2_release_policy[:,:,season])
    if np.log(Qpast) <= Ylevels1[0]: # interpolate between storages at lowest flow level
      SDP1.prescribedR = np.interp(SDP1.simS[year,season],states,SDP1_release_policy[:,0,season])
      SDP2.prescribedR = np.interp(SDP2.simS[year,season],states,SDP2_release_policy[:,0,season])
    elif np.log(Qpast) >= Ylevels1[-1]: # interpolate between storages at highest flow level
      SDP1.prescribedR = np.interp(SDP1.simS[year,season],states,SDP1_release_policy[:,-1,season])
      SDP2.prescribedR = np.interp(SDP2.simS[year,season],states,SDP2_release_policy[:,-1,season])
    else: # interpolate over 2-D grid
      SDP1.prescribedR = f1(np.array([SDP1.simS[year,season],np.log(Qpast)]))[0]
      SDP2.prescribedR = f2(np.array([SDP2.simS[year,season],np.log(Qpast)]))[0]

    # find actual release (what is physically possible) and calculate storage from mass balance
    DDP.getSimStates(Qnow, year, season)
    SDP1.getSimStates(Qnow, year, season)
    SDP2.getSimStates(Qnow, year, season)

    # update past flow for next season's calculation
    Qpast = Qnow

  # calculate total cost (squared deviations from targets) in simulated year
  DDP.getSimCosts(year)
  SDP1.getSimCosts(year)
  SDP2.getSimCosts(year)

## Part g (10 pts)

Based on your simulation from part (f), make a 2x2 panel figure of the empirical cumulative distribution function of percent deviations from the storage, salinity, and wetland flow targets, as well as the sum across all three. Do this for the DDP and two SDP policies using a different color for each and include a legend. $\color{red}{\text{Use the code chunk below to make your plot.}}$

## Part h (10 pts)

Make a bar chart of the number of violations of the minimum irrigation flow and maximum release from the DDP and two SDP policies. $\color{red}{\text{Use the code chunk below to make your plot.}}$

## Part i (10 pts)

Explain what you see in the previous two figures. How does the performance differ on the costs and violations between the three policies and why? $\color{red}{\text{Write your answer in the text chunk below.}}$